# d2_rl — Direction 2: per-air-handler policy, cloned from the MPC teacher, then PPO on the replica

`hvac_controller/d2/rl.py`: behaviour cloning on every teacher label found in the inputs (the `d2_teacher` outputs),
then PPO on the replica caches of dataset `buildings` (never the shipped buildings, which are the evaluation set).
Checkpoints: `/kaggle/working/bc.pt` (cloned) and `/kaggle/working/ppo/last.pt` (after PPO, saved every iteration).
Evaluate both with `d2_eval` (`POLICIES = "d2r:/kaggle/input/<this notebook>/bc.pt,d2r:.../ppo/last.pt"`).

**Inputs:** dataset `buildings`, the `d2_teacher` outputs. CPU (4 processes roll out, the learner is small).
**Internet on.**

In [ ]:
REPO_URL = "https://github.com/CryAndRRich/hvac-controller"
REPO_REF = "direction2"
BC_EPOCHS = 60
PPO_MINUTES = 600      # leave ~1 h of the 12 h session for setup and saving
ROWS, STEPS, PER_ITER = 8, 2880, 16
LR = 1e-4
INIT = ""              # "": clone first; or a path to a .pt to continue from

In [ ]:
import json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path

REPO, INPUT, WORK = Path("/tmp/hvac-controller"), Path("/kaggle/input"), Path("/tmp/work")
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--quiet", REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--quiet", REPO_REF], check=True)
print(subprocess.run(["git", "-C", str(REPO), "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
sys.path.insert(0, str(REPO))
ENV = dict(os.environ, PYTHONPATH=str(REPO), OMP_NUM_THREADS="1", OPENBLAS_NUM_THREADS="1", MKL_NUM_THREADS="1")


def find(pattern):
    """The folder under /kaggle/input that holds a file matching pattern (datasets and notebook outputs alike)."""
    hits = sorted(INPUT.rglob(pattern))
    if not hits:
        raise FileNotFoundError(f"no {pattern} under {INPUT}: attach the input")
    return hits[0].parent

CACHES = find("building_*__*.pkl")
print("caches:", CACHES, len(list(CACHES.glob("*.pkl"))))
DATA = WORK / "labels"
DATA.mkdir(parents=True, exist_ok=True)
for z in sorted(INPUT.rglob("labels_*.zip")):
    with zipfile.ZipFile(z) as f:
        f.extractall(DATA / z.stem)
for p in sorted(INPUT.rglob("*__teacher.npz")) + sorted(INPUT.rglob("*__dagger.npz")):
    shutil.copy(p, DATA / p.name)
print(len(list(DATA.rglob("*.npz"))), "labelled scenarios")

In [ ]:
def run(cmd):
    t0 = time.time()
    proc = subprocess.Popen(cmd, cwd=REPO, env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        if line.startswith("[") or "Error" in line or "Traceback" in line or line.startswith("{"):
            print(line, end="", flush=True)
    print("exit", proc.wait(), f"after {(time.time() - t0) / 60:.0f} min")

init = INIT
if not init:
    run([sys.executable, "-m", "hvac_controller.d2.rl", "bc", "/kaggle/working/bc.pt", "--data", str(DATA),
         "--epochs", str(BC_EPOCHS)])
    init = "/kaggle/working/bc.pt"
run([sys.executable, "-m", "hvac_controller.d2.rl", "train", "/kaggle/working/ppo", "--caches", str(CACHES),
     "--init", init, "--minutes", str(PPO_MINUTES), "--rows", str(ROWS), "--steps", str(STEPS),
     "--per-iter", str(PER_ITER), "--procs", "4", "--lr", str(LR)])